In [0]:
from pyspark.sql import SparkSession

spark

In [0]:
df = (
    spark.readStream
         .format("kafka")
)

In [0]:
kafka_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka-devanshi-joshidevanshi1012-1bdd.e.aivencloud.com:23391")
        .option("subscribe", "openstack-logs")
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
        .option(
            "kafka.sasl.jaas.config",
            'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username="avnadmin" '
            'password="AIVEN_SECRET_REMOVED";'
        )
        .load()
)

In [0]:
!pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.1/614.1 kB 9.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS `log-analytics`.bronze.bronze_logs (
    key STRING,
    value STRING,
    kafka_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
display(
    kafka_df.selectExpr(
        "CAST(key AS STRING)",
        "CAST(value AS STRING)",
        "timestamp"
    ),
    checkpointLocation="/Volumes/log-analytics/bronze/checkpoint_volume/checkpoints"
)

Checkpointing to /Volumes/log-analytics/bronze/checkpoint_volume/checkpoints


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7945231402467370>, line 1
----> 1 display(
      2     kafka_df.selectExpr(
      3         "CAST(key AS STRING)",
      4         "CAST(value AS STRING)",
      5         "timestamp"
      6     ),
      7     checkpointLocation="/Volumes/log-analytics/bronze/checkpoint_volume/checkpoints"
      8 )

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     92     raise type(
     93         e
     94     )("IPython shell

In [0]:
bronze_stream = (
    kafka_df
    .selectExpr(
        "CAST(key AS STRING) as key",
        "CAST(value AS STRING) as value",
        "timestamp as kafka_timestamp"
    )
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)   # <-- Add this line
    .option(
        "checkpointLocation",
        "/Volumes/log-analytics/bronze/checkpoint_volume/checkpoints/bronze_logs"
    )
    .toTable("`log-analytics`.bronze.bronze_logs")
)

In [0]:
from kafka import KafkaProducer

import time

producer = KafkaProducer(
    bootstrap_servers="kafka-devanshi-joshidevanshi1012-1bdd.e.aivencloud.com:23391",
    security_protocol="SASL_SSL",
    sasl_mechanism="SCRAM-SHA-256",
    sasl_plain_username="avnadmin",
    sasl_plain_password="AIVEN_SECRET_REMOVED",
    ssl_cafile="ca.pem",
)

try:
    while True:
        key = "hello"
        value = "world"

        producer.send("openstack-logs", value.encode(), key.encode())

        print(f"Produced: key={key}, value={value} into topic openstack-logs", flush=True)

        time.sleep(1)

finally:
    producer.close()


---------------------------------------------------------------------------
KafkaTimeoutError                         Traceback (most recent call last)
File <command-8953811808215496>, line 5
      1 from kafka import KafkaProducer
      3 import time
----> 5 producer = KafkaProducer(
      6     bootstrap_servers="kafka-devanshi-joshidevanshi1012-1bdd.e.aivencloud.com:23391",
      7     security_protocol="SASL_SSL",
      8     sasl_mechanism="SCRAM-SHA-256",
      9     sasl_plain_username="avnadmin",
     10     sasl_plain_password="AIVEN_SECRET_REMOVED",
     11     ssl_cafile="ca.pem",
     12 )
     14 try:
     15     while True:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-ac730cce-ed86-4b49-b140-722dbf397a82/lib/python3.12/site-packages/kafka/producer/kafka.py:534, in KafkaProducer.__init__(self, **configs)
    530 # We currently depend on eager-resolution of api_version.
    531 # If it wasn't provided as a config option, we need to bootstrap
    532 # to get it.
    533